# Field Boundary Detection — Raw Pixels to Vector Inventory

**For an agriculture customer evaluating Wherobots RasterFlow.**

Crop co-ops, insurers, input retailers, and precision-ag platforms all
need the same thing: a current, complete polygon inventory of every
cultivated field in their service area. Buying this from a vendor is
expensive, patchy, and stale. Building it yourself used to mean a team
of ML engineers and a GPU cluster.

This notebook shows the full RasterFlow path for an ag customer:

1. **Pick an area of interest** — a county, a territory, a customer's
   entire book of business.
2. **Run Fields of the World (FTW)** — a benchmark segmentation model
   trained on Sentinel-2 imagery from 24 countries. RasterFlow handles
   the whole pipeline: imagery fetch → seasonal mosaic → inference.
3. **Vectorize** — convert pixel-class rasters into field polygons with
   confidence scores.
4. **Query & clean** — Sedona SQL to validate geometries, compute
   acreage, filter by score.
5. **Persist & ship** — save as a managed Iceberg table, export GeoJSON
   for the customer's map.

The demo AOI is **Haskell County, Kansas** — ~500 k acres of High Plains
pivot-irrigation cropland. A production deployment scales the same
workflow to multi-state or national footprints without code changes.

## 1. Setup

Two clients: `SedonaContext` for catalog SQL and GeoParquet I/O,
`RasterflowClient` for model orchestration.

In [ ]:
from datetime import datetime
import os, json

import geopandas as gpd
import wkls
from sedona.spark import *
import pyspark.sql.functions as f

from rasterflow_remote import RasterflowClient
from rasterflow_remote.data_models import ModelRecipes, VectorizeMethodEnum

config = SedonaContext.builder().getOrCreate()
sedona = SedonaContext.create(config)
rf_client = RasterflowClient()

TARGET_DATABASE = "ag_field_inventory_demo"
TARGET_TABLE    = "haskell_ks_fields"
TARGET_FQN      = f"org_catalog.{TARGET_DATABASE}.{TARGET_TABLE}"

## 2. Pick the AOI

Customers point RasterFlow at any polygon — a county, a service
territory, a list of leased fields. Here we pull Haskell County, KS
from the `wkls` convenience library and land it in the user's S3 space
in GeoParquet so RasterFlow can read it.

In [ ]:
aoi_gdf = gpd.read_file(wkls["us"]["ks"]["Haskell County"].geojson())
aoi_path = os.getenv("USER_S3_PATH") + "haskell.parquet"
aoi_gdf.to_parquet(aoi_path)

min_lon, min_lat, max_lon, max_lat = aoi_gdf.total_bounds
print(f"AOI: Haskell County, KS")
print(f"  bbox: ({min_lon:.4f}, {min_lat:.4f}) to ({max_lon:.4f}, {max_lat:.4f})")
print(f"  parquet: {aoi_path}")

## 3. Run Fields of the World on Sentinel-2

`build_and_predict_mosaic_recipe` does everything in one call:

- Fetch Sentinel-2 L2A scenes that intersect the AOI and date range.
- Build cloud-screened planting/harvest seasonal median mosaics
  (Red / Green / Blue / NIR bands) — the inputs FTW was trained on.
- Run inference; write a Zarr store with per-pixel probabilities for
  three classes: `non_field_background`, `field`, `field_boundaries`.

The customer writes no ML code, provisions no GPUs. Runtime for one
county is **~30 minutes**; the same call scales to multi-state AOIs
without change.

In [ ]:
model_output_index = rf_client.build_and_predict_mosaic_recipe(
    aoi          = aoi_path,
    start        = datetime(2023, 1, 1),
    end          = datetime(2024, 1, 1),
    crs_epsg     = 3857,                  # Web Mercator for downstream tiling
    model_recipe = ModelRecipes.FTW,
)

model_output_store = model_output_index.first_row_mosaic
print(f"Zarr store: {model_output_store}")
print(f"Index URI : {model_output_index.uri}")

## 4. Vectorize — Pixels to Polygons

`vectorize_mosaic` threshold-masks the probability raster and traces
polygon boundaries, yielding a GeoParquet file of field + boundary
features with a `score_mean` confidence per geometry. Runtime ~5 min
for the county.

In [ ]:
vectorized = rf_client.vectorize_mosaic(
    store            = model_output_store,
    features         = ["field", "field_boundaries"],
    threshold        = 0.5,
    vectorize_method = VectorizeMethodEnum.SEMANTIC_SEGMENTATION_RASTERIO,
    vectorize_config = {"stats": True, "medial_skeletonize": False},
)

print(f"Vectorized output: {vectorized.uri}")

## 5. Load, Validate, and Clean the Polygons

Bring the GeoParquet into Sedona, fix self-intersections, set CRS,
reduce coordinate precision, and simplify. These are one-liners at
scale — the same pipeline runs on a million polygons.

In [ ]:
raw_df = sedona.read.format("geoparquet").load(vectorized.uri) \
    .withColumnRenamed("label", "layer")

TOLERANCE      = 0.00001     # ~1 m in lat/lon
DECIMAL_PLACES = 6           # ~10 cm

cleaned_df = (
    raw_df
    .filter("score_mean >= 0.5")
    .withColumn("geometry", f.expr("ST_MakeValid(geometry)"))
    .withColumn("geometry", f.expr("ST_SetSRID(geometry, 4326)"))
    .withColumn("geometry", f.expr(f"ST_ReducePrecision(geometry, {DECIMAL_PLACES})"))
    .withColumn("geometry", f.expr(f"ST_SimplifyPreserveTopology(geometry, {TOLERANCE})"))
    .cache()
)
cleaned_df.createOrReplaceTempView("raw_fields")

print(f"Features after cleanup: {cleaned_df.count():,}")
cleaned_df.groupBy("layer").count().show()

## 6. Customer-Ready Field Inventory

Compute the numbers an agronomist or underwriter actually needs —
field count, acreage, average size, confidence distribution.

In [ ]:
fields_df = sedona.sql("""
    SELECT
        ROW_NUMBER() OVER (ORDER BY ST_Area(geometry) DESC) AS field_id,
        score_mean                                          AS confidence,
        ROUND(
            ST_Area(ST_Transform(geometry, 'EPSG:4326', 'EPSG:3857'))
            / 4046.86, 2
        ) AS acres,
        geometry
    FROM raw_fields
    WHERE layer = 'field'
""").cache()
fields_df.createOrReplaceTempView("fields")

sedona.sql("""
    SELECT
        COUNT(*)                             AS field_count,
        ROUND(SUM(acres), 0)                 AS total_acres,
        ROUND(AVG(acres), 1)                 AS avg_field_acres,
        ROUND(PERCENTILE(acres, 0.5), 1)     AS median_field_acres,
        ROUND(MAX(acres), 1)                 AS largest_field_acres,
        ROUND(AVG(confidence), 3)            AS avg_confidence
    FROM fields
""").show(truncate=False)

# Acreage size distribution
sedona.sql("""
    SELECT
        CASE
            WHEN acres <   10 THEN 'a) <10 ac (small)'
            WHEN acres <   40 THEN 'b) 10-40 ac'
            WHEN acres <  160 THEN 'c) 40-160 ac (quarter-section)'
            WHEN acres <  640 THEN 'd) 160-640 ac (section)'
            ELSE                   'e) >640 ac (large pivot/fallow)'
        END AS size_bucket,
        COUNT(*)               AS field_count,
        ROUND(SUM(acres), 0)   AS bucket_acres
    FROM fields
    GROUP BY size_bucket
    ORDER BY size_bucket
""").show(truncate=False)

## 7. Persist as a Managed Iceberg Table

The customer's analysts can now query the inventory from any Spark / SQL
tool that reads Iceberg — no handoff, no format conversion.

In [ ]:
sedona.sql(f"CREATE DATABASE IF NOT EXISTS org_catalog.{TARGET_DATABASE}")

fields_df.writeTo(TARGET_FQN).createOrReplace()

print(f"Wrote {TARGET_FQN}")
sedona.sql(f"DESCRIBE TABLE {TARGET_FQN}").show(truncate=False)
sedona.sql(f"SELECT COUNT(*) AS rows FROM {TARGET_FQN}").show()

## 8. Export GeoJSON for the Customer's Map

A small sample — top 200 fields by acreage — goes to GeoJSON for the
field-ops dashboard. (For full-county rendering at scale, RasterFlow
also produces PMTiles in one call — out of scope here.)

In [ ]:
sample_df = sedona.sql(f"""
    SELECT
        field_id,
        ROUND(confidence, 3) AS confidence,
        acres,
        ST_AsGeoJSON(geometry) AS geojson
    FROM {TARGET_FQN}
    ORDER BY acres DESC
    LIMIT 200
""")

features = [
    {
        "type": "Feature",
        "properties": {
            "field_id":   int(row.field_id),
            "acres":      float(row.acres),
            "confidence": float(row.confidence),
        },
        "geometry": json.loads(row.geojson),
    }
    for row in sample_df.collect()
]
fc = {"type": "FeatureCollection", "features": features}

out_path = "/tmp/haskell_ks_fields_top200.geojson"
with open(out_path, "w") as fh:
    json.dump(fc, fh)

print(f"Wrote {len(features)} fields to {out_path}")

## 9. Preview on a Map

In [ ]:
SedonaKepler.create_map(
    fields_df.select("field_id", "acres", "confidence", "geometry"),
    name="Haskell County Field Inventory"
)

---

## What the customer got

| Input | Output |
|---|---|
| One polygon (Haskell County, KS) | Thousands of field polygons with confidence + acreage |
| One date range (2023) | Cloud-screened seasonal Sentinel-2 mosaic, handled by RasterFlow |
| Two API calls (`build_and_predict_mosaic_recipe`, `vectorize_mosaic`) | Iceberg table + GeoJSON in ~35 minutes |

## Scaling up

| Demo | Production |
|---|---|
| 1 county AOI | Multi-state shape collection — same call |
| 1 year window | Nightly refresh via scheduled job |
| Top-200 GeoJSON sample | Full PMTiles output via `vtiles.generate_pmtiles` |
| `org_catalog.ag_field_inventory_demo` | Customer's own managed catalog with IAM / Unity / Glue |

## References
- Kerner et al. 2024, *Fields of The World: A Machine Learning Benchmark Dataset For Global Agricultural Field Boundary Segmentation.* arXiv:2409.16252. Accepted AAAI-2025 AISI track.
- ESA 2015, *Sentinel-2 User Handbook.*